# 河川距離と周辺河川流量の冗長性検証：二段階LightGBM Spatial OOF

## 目的
`distance_to_reliable_river` があるとき、50km・100km周辺のGloFAS流量が追加情報を持つかを検証する。

- 人口変数は説明変数に入れない。
- crop potentialはraw値を使う。
- 灌漑の符号付き差分、存在・欠測フラグは入れない。
- 市場アクセスの周辺変数は入れない。
- 河川関連以外の変数、サンプル、Spatial OOF分割、LightGBM設定を全モデルで固定する。
- 判断材料はOOF R²・RMSE・MAE、残差Moran's I、空間ブロックbootstrapとする。

## 比較モデル
1. 河川変数なし
2. 一定流量以上の河川までの距離のみ
3. 距離＋局所GloFAS
4. 距離＋局所GloFAS＋50km周辺GloFAS
5. 距離＋局所GloFAS＋100km周辺GloFAS
6. 距離＋局所GloFAS＋50km＋100km周辺GloFAS
7. 6＋50km・100km周辺河川距離（現行のfull river WX）


## 1. 設定・データ読み込み
上から順番に実行する。重い計算はセル5のSpatial OOFで行う。

In [ ]:
from __future__ import annotations

import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.neighbors import BallTree
from lightgbm import LGBMClassifier, LGBMRegressor

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ROOT = Path("C:/masterresearch/Comparative_advantage")
GAEZ_DIR = ROOT / "GAEZ"
HYDE_DIR = ROOT / "HYDE3.4"
CROPLAND_DIR = HYDE_DIR / "cropland_npys"
DIST_DIR = ROOT / "distance_to_cities"
GLOFAS_DIR = ROOT / "GloFAS" / "processed_5min"
FEATURE_CACHE = GAEZ_DIR / "CroplandRegression" / "features_cache"
WX_CACHE_DIR = GAEZ_DIR / "CroplandRegression" / "spatial_wx_comparison"
OUTPUT_DIR = GAEZ_DIR / "CroplandRegression" / "river_wx_ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2024
RANDOM_SEED = 42
PRESENCE_THRESHOLD = 0.01
N_POS_SAMPLE = 120_000
N_ZERO_SAMPLE = 120_000
N_SPLITS = 5
N_JOBS = 4
MORAN_K = 8
MORAN_MAX_N = 50_000
BOOTSTRAP_REPS = 500
R2_PARSIMONY_TOLERANCE = 0.001
REFERENCE_MODEL = "distance_plus_local_flow"

def load_cache(*names):
    for name in names:
        path = FEATURE_CACHE / name
        if path.exists():
            print("load cache:", path.name)
            return np.load(path, mmap_mode="r")
    raise FileNotFoundError("Required cache not found: " + ", ".join(names))

def safe_log1p(values):
    values = np.asarray(values, dtype=np.float32)
    values = np.where(np.isfinite(values) & (values > 0), values, 0.0)
    return np.log1p(values).astype(np.float32)

def positive_raw(values):
    values = np.asarray(values, dtype=np.float32)
    return np.where(np.isfinite(values) & (values > 0), values, 0.0).astype(np.float32)

lat = np.load(CROPLAND_DIR / "lat.npy")
lon = np.load(CROPLAND_DIR / "lon.npy")
shape = (len(lat), len(lon))

cropland_cube = np.load(CROPLAND_DIR / "cropland_fraction_1950_2024.npy", mmap_mode="r")
years = np.load(CROPLAND_DIR / "years.npy")
year_index = int(np.where(years == YEAR)[0][0])
cropland = np.asarray(cropland_cube[year_index], dtype=np.float32).copy()
cropland[~np.isfinite(cropland)] = np.nan

# populationは元分析と同じland maskを作るためだけに使い、説明変数には入れない。
population = load_cache("population_density_2024.npy")
elevation = load_cache("elevation_5min.npy")
slope = load_cache("slope_5min.npy")
exclusion = load_cache("exclusion_5min_mode.npy")
city_time = np.load(DIST_DIR / "cities_10_1_12deg_min.npy", mmap_mode="r")
port_time = np.load(DIST_DIR / "ports_05_1_12deg_min.npy", mmap_mode="r")
glofas = np.load(GLOFAS_DIR / "p10_discharge_max_5min_2020.npy", mmap_mode="r")
river_distance = np.load(GLOFAS_DIR / "distance_to_reliable_river_p10_gt_10_m3s_km_5min_2020.npy", mmap_mode="r")
rainfed_value = load_cache("rainfed_value_top5_usd_per_ha_checked_36crops.npy", "rainfed_value_top5_checked_36crops.npy")
irrigated_value = load_cache("irrigated_value_top5_usd_per_ha_checked_36crops.npy", "irrigated_value_top5_checked_36crops.npy")
rainfed_calorie = load_cache("rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy", "rainfed_calorie_top5_checked_36crops.npy")
irrigated_calorie = load_cache("irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy", "irrigated_calorie_top5_checked_36crops.npy")

print("grid shape:", shape)
print("output:", OUTPUT_DIR)


## 2. 共通サンプルと候補変数を作成
全モデルで完全に同じ24万セル候補、同じ欠損除外、同じSpatial foldを使う。

In [ ]:
land_mask = (
    np.isfinite(elevation)
    & np.isfinite(cropland)
    & np.isfinite(population)
    & (population >= 0)
)
presence_raster = land_mask & (cropland > PRESENCE_THRESHOLD)
target_prior = float(presence_raster.sum() / land_mask.sum())

rng = np.random.default_rng(RANDOM_SEED)
positive_flat = np.flatnonzero(presence_raster.ravel())
zero_flat = np.flatnonzero((land_mask & ~presence_raster).ravel())
positive_sample = rng.choice(positive_flat, min(N_POS_SAMPLE, len(positive_flat)), replace=False)
zero_sample = rng.choice(zero_flat, min(N_ZERO_SAMPLE, len(zero_flat)), replace=False)
sample_flat = np.concatenate([positive_sample, zero_sample])
rng.shuffle(sample_flat)
rows, cols = np.unravel_index(sample_flat, shape)

def take(array):
    return np.asarray(array[rows, cols])

rainfed_value_sample = take(rainfed_value).astype(np.float32)
irrigated_value_sample = take(irrigated_value).astype(np.float32)
rainfed_calorie_sample = take(rainfed_calorie).astype(np.float32)
irrigated_calorie_sample = take(irrigated_calorie).astype(np.float32)

value_pair_valid = np.isfinite(rainfed_value_sample) & np.isfinite(irrigated_value_sample)
calorie_pair_valid = np.isfinite(rainfed_calorie_sample) & np.isfinite(irrigated_calorie_sample)
irrigation_value_gain = np.where(
    value_pair_valid, np.maximum(irrigated_value_sample - rainfed_value_sample, 0.0), 0.0
).astype(np.float32)
irrigation_calorie_gain = np.where(
    calorie_pair_valid, np.maximum(irrigated_calorie_sample - rainfed_calorie_sample, 0.0), 0.0
).astype(np.float32)

sample = pd.DataFrame({
    "row": rows.astype(np.int32),
    "col": cols.astype(np.int32),
    "lat": lat[rows].astype(np.float32),
    "lon": lon[cols].astype(np.float32),
    "cropland_fraction": take(cropland).astype(np.float32),
    "presence": (take(cropland) > PRESENCE_THRESHOLD).astype(np.uint8),
    "elevation_m": take(elevation).astype(np.float32),
    "slope": take(slope).astype(np.float32),
    "exclusion_class": np.nan_to_num(take(exclusion), nan=-1).astype(np.int16),
    "log_city_time_20k_min": safe_log1p(take(city_time)),
    "log_port_time_any_min": safe_log1p(take(port_time)),
    "log_glofas_p10_2020": safe_log1p(take(glofas)),
    "log_distance_river_gt10_2020": safe_log1p(take(river_distance)),
    "rainfed_value_top5_raw": positive_raw(rainfed_value_sample),
    "rainfed_calorie_top5_raw": positive_raw(rainfed_calorie_sample),
    "irrigation_value_gain_top5_raw": irrigation_value_gain,
    "irrigation_calorie_gain_top5_raw": irrigation_calorie_gain,
})

wx_names = [
    "wx_50km_rainfed_value_top5_raw",
    "wx_100km_rainfed_value_top5_raw",
    "wx_50km_log_glofas_p10_2020",
    "wx_100km_log_glofas_p10_2020",
    "wx_50km_log_distance_river_gt10_2020",
    "wx_100km_log_distance_river_gt10_2020",
]
for name in wx_names:
    path = WX_CACHE_DIR / f"{name}.npy"
    if not path.exists():
        raise FileNotFoundError(f"WX cache is missing: {path}")
    raster = np.load(path, mmap_mode="r")
    sample[name] = np.asarray(raster[rows, cols], dtype=np.float32)

sample["spatial_block"] = (
    np.floor((sample["lat"] + 90) / 10).astype(int) * 36
    + np.floor((sample["lon"] + 180) / 10).astype(int)
)

NON_RIVER_FEATURES = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "rainfed_value_top5_raw",
    "rainfed_calorie_top5_raw",
    "irrigation_value_gain_top5_raw",
    "irrigation_calorie_gain_top5_raw",
    "wx_50km_rainfed_value_top5_raw",
    "wx_100km_rainfed_value_top5_raw",
]

MODEL_FEATURES = {
    "no_river": NON_RIVER_FEATURES,
    "distance_only": NON_RIVER_FEATURES + ["log_distance_river_gt10_2020"],
    "distance_plus_local_flow": NON_RIVER_FEATURES + [
        "log_distance_river_gt10_2020", "log_glofas_p10_2020"
    ],
    "distance_plus_50km_flow": NON_RIVER_FEATURES + [
        "log_distance_river_gt10_2020", "log_glofas_p10_2020",
        "wx_50km_log_glofas_p10_2020",
    ],
    "distance_plus_100km_flow": NON_RIVER_FEATURES + [
        "log_distance_river_gt10_2020", "log_glofas_p10_2020",
        "wx_100km_log_glofas_p10_2020",
    ],
    "distance_plus_50_100km_flow": NON_RIVER_FEATURES + [
        "log_distance_river_gt10_2020", "log_glofas_p10_2020",
        "wx_50km_log_glofas_p10_2020", "wx_100km_log_glofas_p10_2020",
    ],
    "current_full_river_wx": NON_RIVER_FEATURES + [
        "log_distance_river_gt10_2020", "log_glofas_p10_2020",
        "wx_50km_log_glofas_p10_2020", "wx_100km_log_glofas_p10_2020",
        "wx_50km_log_distance_river_gt10_2020",
        "wx_100km_log_distance_river_gt10_2020",
    ],
}

all_features = sorted(set().union(*MODEL_FEATURES.values()))
sample = (
    sample.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=all_features + ["cropland_fraction", "presence"])
    .reset_index(drop=True)
)

if sample["presence"].nunique() != 2:
    raise ValueError("Both presence classes are required.")

groups = sample["spatial_block"].to_numpy()
y_presence = sample["presence"].to_numpy(dtype=np.uint8)
splits = list(GroupKFold(n_splits=N_SPLITS).split(sample, y_presence, groups=groups))

try:
    area_raster = np.load(CROPLAND_DIR / "grid_area_km2.npy", mmap_mode="r")
    sample_area = np.asarray(
        area_raster[sample["row"].to_numpy(int), sample["col"].to_numpy(int)],
        dtype=float,
    )
    n_pos_population = int(presence_raster.sum())
    n_zero_population = int(land_mask.sum() - presence_raster.sum())
    n_pos_sample = int((y_presence == 1).sum())
    n_zero_sample = int((y_presence == 0).sum())
    area_weight = np.where(
        y_presence == 1,
        n_pos_population / max(n_pos_sample, 1),
        n_zero_population / max(n_zero_sample, 1),
    ) * sample_area
except FileNotFoundError:
    print("grid_area_km2.npy not found; area weighting is skipped.")
    area_weight = None

print("common analysis rows:", len(sample))
print("presence share in sample:", sample["presence"].mean())
print("population target prior:", target_prior)
print("spatial blocks:", sample["spatial_block"].nunique())
print()
for model_name, features in MODEL_FEATURES.items():
    print(f"{model_name:34s} {len(features):2d} features")


## 3. 河川候補変数どうしの相関
50km・100kmがほぼ同じ情報かを、共通サンプル上のSpearman相関で確認する。

In [ ]:
RIVER_CANDIDATES = [
    "log_distance_river_gt10_2020",
    "log_glofas_p10_2020",
    "wx_50km_log_glofas_p10_2020",
    "wx_100km_log_glofas_p10_2020",
    "wx_50km_log_distance_river_gt10_2020",
    "wx_100km_log_distance_river_gt10_2020",
]
river_corr = sample[RIVER_CANDIDATES].corr(method="spearman")
display(river_corr.round(3))

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(river_corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(RIVER_CANDIDATES)), RIVER_CANDIDATES, rotation=55, ha="right")
ax.set_yticks(range(len(RIVER_CANDIDATES)), RIVER_CANDIDATES)
for i in range(len(RIVER_CANDIDATES)):
    for j in range(len(RIVER_CANDIDATES)):
        ax.text(j, i, f"{river_corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Spearman correlation among river-related candidate features")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 4. 二段階LightGBMと評価関数
元の人口なしモデルと同じハイパーパラメータ、prior shift補正を使う。

In [ ]:
def adjust_probability_prior_shift(probability, sample_prior, population_prior, eps=1e-6):
    probability = np.clip(np.asarray(probability, dtype=float), eps, 1 - eps)
    sample_prior = float(np.clip(sample_prior, eps, 1 - eps))
    population_prior = float(np.clip(population_prior, eps, 1 - eps))
    odds = probability / (1.0 - probability)
    odds *= (population_prior / (1.0 - population_prior)) / (sample_prior / (1.0 - sample_prior))
    return odds / (1.0 + odds)

def regression_metrics(y_true, y_pred, weights=None):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if weights is None:
        weights = np.ones(len(y_true), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)[valid]
    residual = y_true - y_pred
    observed_mean = np.average(y_true, weights=weights)
    ss_res = np.sum(weights * residual ** 2)
    ss_tot = np.sum(weights * (y_true - observed_mean) ** 2)
    return {
        "n": int(len(y_true)),
        "observed_mean": float(observed_mean),
        "predicted_mean": float(np.average(y_pred, weights=weights)),
        "r2": float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
        "rmse": float(np.sqrt(np.average(residual ** 2, weights=weights))),
        "mae": float(np.average(np.abs(residual), weights=weights)),
        "bias_observed_minus_predicted": float(np.average(residual, weights=weights)),
    }

def fit_models(train_df, features, fold_number):
    classifier = LGBMClassifier(
        objective="binary", n_estimators=450, learning_rate=0.035,
        num_leaves=31, min_child_samples=80, subsample=0.85,
        colsample_bytree=0.85, random_state=RANDOM_SEED + fold_number,
        n_jobs=N_JOBS, verbose=-1,
    )
    classifier.fit(
        train_df[features], train_df["presence"],
        categorical_feature=["exclusion_class"],
    )
    positive_train = train_df[train_df["presence"].eq(1)]
    regressor = LGBMRegressor(
        objective="regression", n_estimators=550, learning_rate=0.03,
        num_leaves=31, min_child_samples=60, subsample=0.85,
        colsample_bytree=0.85, random_state=RANDOM_SEED + fold_number,
        n_jobs=N_JOBS, verbose=-1,
    )
    regressor.fit(
        positive_train[features], positive_train["cropland_fraction"],
        categorical_feature=["exclusion_class"],
    )
    return classifier, regressor


## 5. 全モデルを同じ5-fold Spatial OOFで比較
このセルが最も時間を要する。7モデル × 5fold × 二段階を学習する。

In [ ]:
observed = sample["cropland_fraction"].to_numpy(dtype=float)
prediction_by_model = {}
fold_rows = []

for model_number, (model_name, features) in enumerate(MODEL_FEATURES.items(), start=1):
    print("=" * 80)
    print(f"MODEL {model_number}/{len(MODEL_FEATURES)}: {model_name} ({len(features)} features)")
    print("=" * 80)
    expected_oof = np.full(len(sample), np.nan, dtype=np.float32)

    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        print(f"  fold {fold_number}/{N_SPLITS}")
        train_df = sample.iloc[train_idx]
        test_df = sample.iloc[test_idx]
        classifier, regressor = fit_models(train_df, features, fold_number)

        raw_probability = classifier.predict_proba(test_df[features])[:, 1]
        calibrated_probability = adjust_probability_prior_shift(
            raw_probability, float(train_df["presence"].mean()), target_prior
        )
        conditional_fraction = np.clip(regressor.predict(test_df[features]), 0.0, 1.0)
        expected_fraction = calibrated_probability * conditional_fraction
        expected_oof[test_idx] = expected_fraction.astype(np.float32)

        y_test_presence = test_df["presence"].to_numpy(dtype=np.uint8)
        y_test_fraction = test_df["cropland_fraction"].to_numpy(dtype=float)
        positive_test = y_test_presence == 1
        combined = regression_metrics(y_test_fraction, expected_fraction)
        conditional = regression_metrics(
            y_test_fraction[positive_test], conditional_fraction[positive_test]
        )
        fold_rows.append({
            "model": model_name,
            "fold": fold_number,
            "n_features": len(features),
            "combined_r2": combined["r2"],
            "combined_rmse": combined["rmse"],
            "combined_mae": combined["mae"],
            "conditional_r2": conditional["r2"],
            "conditional_rmse": conditional["rmse"],
            "presence_auc": float(roc_auc_score(y_test_presence, raw_probability)),
            "presence_average_precision": float(average_precision_score(y_test_presence, raw_probability)),
        })
        del classifier, regressor, raw_probability, calibrated_probability
        del conditional_fraction, expected_fraction
        gc.collect()

    if not np.isfinite(expected_oof).all():
        raise RuntimeError(f"OOF prediction is incomplete: {model_name}")
    prediction_by_model[model_name] = expected_oof

fold_metrics = pd.DataFrame(fold_rows)
global_rows = []
weight_specs = {"unweighted_sample": None}
if area_weight is not None:
    weight_specs["area_weighted_reweighted"] = area_weight

for model_name, prediction in prediction_by_model.items():
    for weighting, weights in weight_specs.items():
        global_rows.append({
            "model": model_name,
            "weighting": weighting,
            "n_features": len(MODEL_FEATURES[model_name]),
            **regression_metrics(observed, prediction, weights=weights),
        })
global_metrics = pd.DataFrame(global_rows)

prediction_table = sample[["row", "col", "lat", "lon", "spatial_block", "presence", "cropland_fraction"]].copy()
for model_name, prediction in prediction_by_model.items():
    prediction_table[f"pred_{model_name}"] = prediction

fold_metrics.to_csv(OUTPUT_DIR / "river_wx_ablation_fold_metrics.csv", index=False, encoding="utf-8-sig")
global_metrics.to_csv(OUTPUT_DIR / "river_wx_ablation_global_metrics.csv", index=False, encoding="utf-8-sig")
prediction_table.to_csv(OUTPUT_DIR / "river_wx_ablation_oof_predictions.csv.gz", index=False, compression="gzip")
print("OOF completed.")


## 6. OOF指標と簡潔性の判断
最良R²との差が0.001以内なら、より少ない変数のモデルも実質同等候補として表示する。

In [ ]:
primary = (
    global_metrics[global_metrics["weighting"].eq("unweighted_sample")]
    .copy()
    .sort_values("r2", ascending=False)
)
reference_row = primary[primary["model"].eq(REFERENCE_MODEL)].iloc[0]
primary["delta_r2_vs_reference"] = primary["r2"] - reference_row["r2"]
primary["delta_rmse_vs_reference"] = primary["rmse"] - reference_row["rmse"]
primary["delta_mae_vs_reference"] = primary["mae"] - reference_row["mae"]
display(primary[[
    "model", "n_features", "r2", "rmse", "mae",
    "delta_r2_vs_reference", "delta_rmse_vs_reference", "delta_mae_vs_reference",
]].round(6))

if area_weight is not None:
    print("AREA-WEIGHTED METRICS")
    display(
        global_metrics[global_metrics["weighting"].eq("area_weighted_reweighted")]
        .sort_values("r2", ascending=False)
        .round(6)
    )

best_r2 = float(primary["r2"].max())
near_best = primary[primary["r2"] >= best_r2 - R2_PARSIMONY_TOLERANCE].copy()
parsimonious = near_best.sort_values(["n_features", "rmse"]).iloc[0]
best_model = primary.iloc[0]["model"]
print("Best OOF R² model       :", best_model)
print("Parsimonious candidate :", parsimonious["model"])
print("Tolerance              :", R2_PARSIMONY_TOLERANCE)

plot_data = primary.sort_values("r2", ascending=True)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, metric, color in zip(axes, ["r2", "rmse", "mae"], ["#4472C4", "#ED7D31", "#70AD47"]):
    ax.barh(plot_data["model"], plot_data[metric], color=color)
    ax.set_title(f"Spatial OOF {metric.upper()}")
    ax.grid(axis="x", alpha=0.2)
plt.show()


## 7. 残差Moran's I
周辺河川変数を加えることで、予測精度だけでなく残差の空間相関が低下するかを確認する。

In [ ]:
def morans_i_knn(frame, residual, k=MORAN_K, max_n=MORAN_MAX_N, seed=RANDOM_SEED + 7000):
    residual = np.asarray(residual, dtype=float)
    coords = frame[["lat", "lon"]].to_numpy(dtype=float)
    valid = np.isfinite(residual) & np.isfinite(coords).all(axis=1)
    indices = np.flatnonzero(valid)
    if len(indices) > max_n:
        rng = np.random.default_rng(seed)
        indices = np.sort(rng.choice(indices, size=max_n, replace=False))
    selected_residual = residual[indices]
    selected_coords = coords[indices]
    tree = BallTree(np.radians(selected_coords), metric="haversine")
    _, neighbors = tree.query(np.radians(selected_coords), k=k + 1)
    neighbors = neighbors[:, 1:]
    centered = selected_residual - selected_residual.mean()
    denominator = float(np.sum(centered ** 2))
    numerator = float(np.sum(centered[:, None] * centered[neighbors]) / neighbors.shape[1])
    return {
        "morans_i": numerator / denominator if denominator > 0 else np.nan,
        "expected_i": -1.0 / (len(indices) - 1),
        "n": len(indices),
        "k": k,
    }

moran_rows = []
for model_name, prediction in prediction_by_model.items():
    residual = observed - prediction
    moran_rows.append({"model": model_name, **morans_i_knn(sample, residual)})
moran_table = pd.DataFrame(moran_rows).sort_values("morans_i")
moran_table.to_csv(OUTPUT_DIR / "river_wx_ablation_residual_morans_i.csv", index=False, encoding="utf-8-sig")
display(moran_table.round(6))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(moran_table["model"], moran_table["morans_i"], color="#5B9BD5")
ax.set_title("Moran's I of Spatial OOF residuals (lower is better)")
ax.set_xlabel("Moran's I")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()


## 8. 空間ブロックbootstrapによるRMSE差の不確実性
10度ブロックを再標本化し、基準モデルとの差を計算する。`delta_rmse < 0`なら基準モデルより改善。95%区間が0をまたぐ場合は、改善が安定しているとは言いにくい。

In [ ]:
def block_bootstrap_rmse_delta(observed, prediction, reference_prediction, block_ids, reps=500, seed=42):
    frame = pd.DataFrame({
        "block": np.asarray(block_ids),
        "se_model": (np.asarray(observed) - np.asarray(prediction)) ** 2,
        "se_reference": (np.asarray(observed) - np.asarray(reference_prediction)) ** 2,
    })
    block_stats = frame.groupby("block", sort=True).agg(
        n=("se_model", "size"),
        sse_model=("se_model", "sum"),
        sse_reference=("se_reference", "sum"),
    )
    n = block_stats["n"].to_numpy(float)
    sse_model = block_stats["sse_model"].to_numpy(float)
    sse_reference = block_stats["sse_reference"].to_numpy(float)
    rng = np.random.default_rng(seed)
    delta = np.empty(reps, dtype=float)
    n_blocks = len(block_stats)
    for rep in range(reps):
        draw = rng.integers(0, n_blocks, size=n_blocks)
        denominator = n[draw].sum()
        rmse_model = np.sqrt(sse_model[draw].sum() / denominator)
        rmse_reference = np.sqrt(sse_reference[draw].sum() / denominator)
        delta[rep] = rmse_model - rmse_reference
    return {
        "delta_rmse_mean": float(delta.mean()),
        "ci_low_2_5pct": float(np.quantile(delta, 0.025)),
        "ci_high_97_5pct": float(np.quantile(delta, 0.975)),
        "probability_better": float(np.mean(delta < 0)),
    }

reference_prediction = prediction_by_model[REFERENCE_MODEL]
bootstrap_rows = []
for model_name, prediction in prediction_by_model.items():
    bootstrap_rows.append({
        "model": model_name,
        **block_bootstrap_rmse_delta(
            observed, prediction, reference_prediction,
            sample["spatial_block"].to_numpy(),
            reps=BOOTSTRAP_REPS, seed=RANDOM_SEED + 9000,
        ),
    })
bootstrap_table = pd.DataFrame(bootstrap_rows).sort_values("delta_rmse_mean")
bootstrap_table.to_csv(OUTPUT_DIR / "river_wx_ablation_block_bootstrap.csv", index=False, encoding="utf-8-sig")
display(bootstrap_table.round(6))


## 9. 基準モデルと比較モデルの残差地図
最良モデルが基準モデルと同じ場合は、周辺河川変数をすべて入れた現行型と比較する。赤い改善図は、比較モデルの絶対誤差が基準より小さい場所を示す。

In [ ]:
comparison_model = best_model if best_model != REFERENCE_MODEL else "current_full_river_wx"
reference_prediction = prediction_by_model[REFERENCE_MODEL]
comparison_prediction = prediction_by_model[comparison_model]
reference_residual = observed - reference_prediction
comparison_residual = observed - comparison_prediction
absolute_error_improvement = np.abs(reference_residual) - np.abs(comparison_residual)

residual_limit = max(float(np.nanpercentile(np.abs(np.concatenate([reference_residual, comparison_residual])), 99.5)), 0.01)
improvement_limit = max(float(np.nanpercentile(np.abs(absolute_error_improvement), 99.5)), 0.005)
fig, axes = plt.subplots(2, 2, figsize=(18, 10), constrained_layout=True)
panels = [
    (observed, "Observed cropland fraction", "Greens", 0.0, 1.0),
    (reference_residual, f"Residual: {REFERENCE_MODEL}", "coolwarm", -residual_limit, residual_limit),
    (comparison_residual, f"Residual: {comparison_model}", "coolwarm", -residual_limit, residual_limit),
    (absolute_error_improvement, f"Absolute-error improvement: {comparison_model} vs reference", "coolwarm", -improvement_limit, improvement_limit),
]
for ax, (values, title, cmap, vmin, vmax) in zip(axes.ravel(), panels):
    scatter = ax.scatter(
        sample["lon"], sample["lat"], c=values, s=0.35,
        cmap=cmap, vmin=vmin, vmax=vmax, linewidths=0, rasterized=True,
    )
    ax.set_title(title)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    ax.grid(alpha=0.15)
    fig.colorbar(scatter, ax=ax, fraction=0.046, pad=0.03)
plt.show()

print("Reference model :", REFERENCE_MODEL)
print("Comparison model:", comparison_model)
print("Saved tables     :", OUTPUT_DIR)


## 10. 結果の判断基準

### 周辺GloFASを外してよい場合
- `distance_plus_local_flow`と比較してR²改善がほぼない。
- RMSE差の空間ブロックbootstrap 95%区間が0をまたぐ。
- 残差Moran's Iも明確に低下しない。
この場合、主仕様は河川距離＋局所GloFASとし、50km・100kmは感度分析へ回す。

### 周辺GloFASを残す場合
- OOF R²・RMSEが再現性をもって改善する。
- bootstrapでも改善確率が高く、95%区間が0より小さい。
- 残差Moran's Iが低下する。
この場合も、50kmと100kmの両方が必要か、一方だけで十分かを結果表から選ぶ。

### 周辺河川距離
`current_full_river_wx`が周辺GloFASだけのモデルを改善しないなら、50km・100km周辺河川距離は除外する。
